# Indexing And Retrieval

This notebook focuses on how indexed payloads, chunking, filtering, and paper-scoped retrieval work together.


## Setup And Demo Papers

The example uses two small papers so every retrieved chunk can be inspected directly.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import tempfile


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "episcope").exists():
            return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is not None:
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

WORK_DIR = Path(tempfile.mkdtemp(prefix="episcope-notebook-"))
WORK_DIR


In [ ]:
from episcope.schemas import PaperMetadata, StructuredSection

sample_papers = {
    "paper_open_data": {
        "metadata": PaperMetadata(
            title="Trial data sharing and reuse",
            abstract="A randomized trial reused a public patient-level dataset from a hospital registry.",
            keywords=["trial", "registry", "data sharing"],
        ),
        "sections": [
            StructuredSection(
                title="Methods",
                section_type="Methods",
                content=(
                    "The analysis used patient records from the National Hospital Registry. "
                    "The registry stores admission dates, treatment groups, and mortality outcomes."
                ),
            ),
            StructuredSection(
                title="Data availability",
                section_type="Data availability",
                content=(
                    "De-identified trial data and the analysis code are available from the public repository. "
                    "The dataset can be reused for non-commercial research after registration."
                ),
            ),
        ],
    },
    "paper_closed_data": {
        "metadata": PaperMetadata(
            title="Hospital cohort study",
            abstract="A cohort study collected clinical data directly from participating hospitals.",
            keywords=["cohort", "hospital", "mortality"],
        ),
        "sections": [
            StructuredSection(
                title="Participants",
                section_type="Methods",
                content=(
                    "The cohort included adult patients admitted to three hospitals. "
                    "Data were collected by the study team from electronic health records."
                ),
            ),
            StructuredSection(
                title="Data sharing",
                section_type="Data availability",
                content=(
                    "The patient dataset cannot be shared publicly because the consent agreement "
                    "does not permit redistribution of individual-level hospital records."
                ),
            ),
        ],
    },
}

list(sample_papers)


In [ ]:
import re
from typing import Iterable

import numpy as np

from episcope.rag.embeddings.base import Embedder


class TinyKeywordEmbedder(Embedder):
    vocabulary = (
        "data",
        "dataset",
        "source",
        "cohort",
        "survey",
        "registry",
        "trial",
        "patient",
        "hospital",
        "mortality",
        "covid",
        "treatment",
        "remdesivir",
        "supplement",
        "table",
        "figure",
        "reference",
        "database",
    )

    @property
    def model_name(self) -> str:
        return "tiny-keyword-demo"

    @property
    def dim(self) -> int:
        return len(self.vocabulary)

    def embed_text(self, text: str) -> list[float]:
        text = text.lower()
        counts = []
        for term in self.vocabulary:
            pattern = rf"\b{re.escape(term)}s?\b"
            counts.append(float(len(re.findall(pattern, text))))

        vector = np.array(counts, dtype="float32")
        norm = float(np.linalg.norm(vector))
        if norm:
            vector = vector / norm
        return vector.tolist()

    def embed_texts(self, texts: Iterable[str]) -> list[list[float]]:
        return [self.embed_text(text) for text in texts]


embedder = TinyKeywordEmbedder()
embedder.model_name, embedder.dim


## Build A Local Index

Chunking controls what the retriever can return. Shorter chunks are easier to inspect; larger chunks preserve more surrounding context.


In [ ]:
from episcope.rag.indexing.chunking import FixedSizeChunker, ParagraphChunker

sample_text = sample_papers["paper_open_data"]["sections"][1].content
chunkers = {
    "fixed_size": FixedSizeChunker(chunk_size=90, chunk_overlap=20),
    "paragraph": ParagraphChunker(min_chunk_size=40),
}

for name, chunker in chunkers.items():
    chunks = chunker.chunk(sample_text)
    print(f"{name}: {len(chunks)} chunk(s)")
    for chunk in chunks:
        if len(chunk.split()) < 4:
            continue
        preview = chunk[:85].rsplit(" ", 1)[0]
        if preview != chunk:
            preview = preview + "..."
        print("  -", preview)


In [ ]:
from episcope.rag.indexing.chunking import FixedSizeChunker
from episcope.rag.indexing.indexer import Indexer
from episcope.rag.retrieval.candidates import SemanticCandidateRetriever
from episcope.rag.retrieval.retriever import Retriever
from episcope.vectordb.file import FileDB

vdb = FileDB(str(WORK_DIR / "index"))
indexer = Indexer(
    vdb,
    embedder=embedder,
    chunker=FixedSizeChunker(chunk_size=500, chunk_overlap=50),
)

for paper_id, paper in sample_papers.items():
    indexer.index_paper(paper["sections"], paper["metadata"], paper_id=paper_id)

vdb.save()
semantic_candidates = SemanticCandidateRetriever(vdb, dense_embedder=embedder)
retriever = Retriever(vdb, candidate_retrievers=[semantic_candidates], use_rerank=False)

len(vdb.get_points()), vdb.get_embedding_model()


## Inspect Indexed Payloads

Each vector point keeps the text plus useful metadata such as `paper_id`, `section_title`, and `section_type`. These fields support filtering and provenance.


In [ ]:
for point in vdb.get_points()[:4]:
    print({key: point.get(key) for key in ["paper_id", "section_title", "section_type", "is_metadata"]})
    print(point["text"][:160], "\n")


## Compare Corpus And Paper-Scoped Retrieval

Use `retrieve` to search the whole index and `retrieve_by_paper` when a workflow should only use evidence from one paper.


In [ ]:
query = "patient-level registry data source"
corpus_results = retriever.retrieve(query, top_k=4)
paper_results = retriever.retrieve_by_paper(query, "paper_closed_data", top_k=3)

print("Corpus search")
for result in corpus_results:
    print(f"- {result.paper_id}: {result.section_title} ({result.similarity_score:.3f})")

print("\nWithin paper_closed_data")
for result in paper_results:
    print(f"- {result.paper_id}: {result.section_title} ({result.similarity_score:.3f})")


## Filter By Payload Fields

Filters are useful when a task should search only methods, data availability statements, abstracts, or another payload-defined slice.


In [ ]:
filtered = retriever.retrieve(
    "can the dataset be shared publicly?",
    top_k=5,
    filter={"section_type": "Data availability"},
)

for result in filtered:
    print(f"- {result.paper_id}: {result.section_title}")
    print(result.text[:220], "\n")
